# CAFE: Generalizable Facial Expression Recognition
### Full Training + Evaluation Notebook
Based on ECCV 2024 — Zhang et al.

**Pipeline:**
```
Input image
  ├── CLIP (frozen) ──────────────────── image_features (512)
  └── ResNet-18 (trained) → sigmoid ──── mask
                                    ↓
                    image_features × sigmoid(x)  = masked features
                                    ↓
              channel separation (7 groups) + channel diverse loss
                                    ↓
                    loss = l_cls + 1.5×l_sep + 5×l_div
```

## 1. Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install ftfy regex tqdm kagglehub scipy scikit-learn pandas numpy matplotlib
!pip install git+https://github.com/openai/CLIP.git
!pip install opencv-python-headless

  Preparing metadata (setup.py) ... done


## 2. Kaggle API Setup & Mount Drive (Optional)

**Google Colab인 경우:** Drive 마운트 후 `resnet18_msceleb.pth`와 `kaggle.json`을 Drive에서 불러옵니다.

**기타 Jupyter 환경:** 파일을 직접 업로드하거나 경로를 수정하세요.

In [5]:
import os


FILE_FOLDER = '.'   # 현재 디렉토리에 resnet18_msceleb.pth 배치
SAVE_DIR = '.'

MSCELEB_PATH = os.path.join(FILE_FOLDER, 'resnet18_msceleb.pth')
print('MS-Celeb weights found:', os.path.exists(MSCELEB_PATH))
print('Save dir:', SAVE_DIR)

MS-Celeb weights found: True
Save dir: .


## 3. Dataset Download

### Train Dataset (하나만 주석 해제)

In [ ]:
import kagglehub

# --- RAF-DB ---
_train_raw = kagglehub.dataset_download("shuvoalok/raf-db-dataset")
TRAIN_DATASET_PATH = os.path.join(_train_raw, 'DATASET')
TRAIN_DATASET_NAME = 'RAFDB'
TRAIN_PHASE = 'train'
VALIDATION_PHASE = None

# --- FERPlus ---
# _train_raw = kagglehub.dataset_download("arnabkumarroy02/ferplus")
# TRAIN_DATASET_PATH = _train_raw
# TRAIN_DATASET_NAME = 'FERPlus'
# TRAIN_PHASE = 'train'
# VALIDATION_PHASE = 'validation'

# --- AffectNet ---
# _train_raw = kagglehub.dataset_download("mstjebashazida/affectnet")
# TRAIN_DATASET_PATH = os.path.join(_train_raw, 'archive (3)')
# TRAIN_DATASET_NAME = 'AffectNET'
# TRAIN_PHASE = 'Train'
# VALIDATION_PHASE = None

# --- MMA ---
# _train_raw = kagglehub.dataset_download("mahmoudima/mma-facial-expression")
# TRAIN_DATASET_PATH = os.path.join(_train_raw, 'MMAFEDB')
# TRAIN_DATASET_NAME = 'MMA'
# TRAIN_PHASE = 'train'
# VALIDATION_PHASE = 'valid'

# --- SFEW ---
# _train_raw = kagglehub.dataset_download("vlntnstarodub/datasetsfew")
# TRAIN_DATASET_PATH = _train_raw
# TRAIN_DATASET_NAME = 'SFEW'
# TRAIN_PHASE = 'Train'
# VALIDATION_PHASE = None

print(f'Train: {TRAIN_DATASET_NAME} -> {TRAIN_DATASET_PATH}')
print('Contents:', os.listdir(TRAIN_DATASET_PATH))

100%|██████████| 37.7M/37.7M [00:00<00:00, 92.0MB/s]

Extracting files...


### Test Dataset (하나만 주석 해제)

In [7]:
# --- RAF-DB ---
_test_raw = kagglehub.dataset_download("shuvoalok/raf-db-dataset")
TEST_DATASET_PATH = os.path.join(_test_raw, 'DATASET')
TEST_DATASET_NAME = 'RAFDB'
TEST_PHASE = 'test'

# --- FERPlus ---
# _test_raw = kagglehub.dataset_download("arnabkumarroy02/ferplus")
# TEST_DATASET_PATH = _test_raw
# TEST_DATASET_NAME = 'FERPlus'
# TEST_PHASE = 'test'

# --- AffectNet ---
# _test_raw = kagglehub.dataset_download("mstjebashazida/affectnet")
# TEST_DATASET_PATH = os.path.join(_test_raw, 'archive (3)')
# TEST_DATASET_NAME = 'AffectNET'
# TEST_PHASE = 'Test'

# --- MMA ---
# _test_raw = kagglehub.dataset_download("mahmoudima/mma-facial-expression")
# TEST_DATASET_PATH = os.path.join(_test_raw, 'MMAFEDB')
# TEST_DATASET_NAME = 'MMA'
# TEST_PHASE = 'test'

# --- SFEW ---
# _test_raw = kagglehub.dataset_download("vlntnstarodub/datasetsfew")
# TEST_DATASET_PATH = _test_raw
# TEST_DATASET_NAME = 'SFEW'
# TEST_PHASE = 'Val'

print(f'Test: {TEST_DATASET_NAME} -> {TEST_DATASET_PATH}')
print('Contents:', os.listdir(TEST_DATASET_PATH))

Test: RAFDB -> /root/.cache/kagglehub/datasets/shuvoalok/raf-db-dataset/versions/2/DATASET
Contents: ['test', 'train']


## 4. Imports & CLIP

In [8]:
import cv2
import math
import random
import numpy as np
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
from torchvision import transforms
from torch.autograd import Variable
from torch.nn.modules.module import Module
from torch.nn.modules.utils import _pair
from torch.nn.parameter import Parameter
from torch.utils.data import random_split
from scipy.ndimage import gaussian_filter

import clip
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

clip_model, preprocess = clip.load("ViT-B/32", device=device)
print('CLIP loaded.')

Device: cpu


100%|███████████████████████████████████████| 338M/338M [00:04<00:00, 86.6MiB/s]


CLIP loaded.


## 5. Label Maps

In [ ]:
raf_map = {
    '1': 5, '2': 2, '3': 1, '4': 3, '5': 4, '6': 0, '7': 6
}

ferplus_map = {
    'fear': 2, 'suprise': 5, 'angry': 0, 'neutral': 6,
    'sad': 4, 'disgust': 1, 'happy': 3
}

affectnet_train_map = {
    'anger': 0, 'disgust': 1, 'fear': 2, 'happy': 3,
    'neutral': 6, 'sad': 4, 'surprise': 5
}

affectnet_test_map = {
    'Anger': 0, 'disgust': 1, 'fear': 2, 'happy': 3,
    'neutral': 6, 'sad': 4, 'surprise': 5
}

sfew_map = {
    'Angry': 0, 'Disgust': 1, 'Fear': 2, 'Happy': 3,
    'Neutral': 6, 'Sad': 4, 'Surprise': 5
}

mma_map = {
    'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3,
    'neutral': 6, 'sad': 4, 'surprise': 5
}

TRAIN_MAP_DICT = {
    'RAFDB': raf_map, 'FERPlus': ferplus_map,
    'AffectNET': affectnet_train_map, 'MMA': mma_map, 'SFEW': sfew_map,
}
TEST_MAP_DICT = {
    'RAFDB': raf_map, 'FERPlus': ferplus_map,
    'AffectNET': affectnet_test_map, 'MMA': mma_map, 'SFEW': sfew_map,
}

TRAIN_MAP = TRAIN_MAP_DICT[TRAIN_DATASET_NAME]
TEST_MAP = TEST_MAP_DICT[TEST_DATASET_NAME]

# label index 0~6 (all datasets)
LABEL_NAMES = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']
NUM_CLASSES = 7

print(f'Train: {TRAIN_DATASET_NAME} | Test: {TEST_DATASET_NAME}')

## 6. Dataset & DataLoader

In [ ]:
def add_g(image_array, mean=0.0, var=30):
    std = var ** 0.5
    image_add = image_array + np.random.normal(mean, std, image_array.shape)
    image_add = np.clip(image_add, 0, 255).astype(np.uint8)
    return image_add

def flip_image(image_array):
    return cv2.flip(image_array, 1)

def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True


class FolderDataset(data.Dataset):

    def __init__(self, root, phase='train', transform=None, label_map=None, augment=None):
        self.transform = transform
        self.aug_func  = [flip_image, add_g]
        self.phase     = phase
        self.label_map = label_map
        self.augment   = augment if augment is not None else (phase.lower() == 'train')

        split_dir = os.path.join(root, phase)
        self.classes = sorted([c for c in os.listdir(split_dir)
                               if c not in ('contempt', 'Contempt') and os.path.isdir(os.path.join(split_dir, c))])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        self.file_paths = []
        self.labels     = []

        for cls in self.classes:
            cls_dir = os.path.join(split_dir, cls)
            for fname in sorted(os.listdir(cls_dir)):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.file_paths.append(os.path.join(cls_dir, fname))
                    if self.label_map is not None:
                        mapped_label = self.label_map[cls]
                    else:
                        mapped_label = self.class_to_idx[cls]
                    self.labels.append(mapped_label)

        print(f'[{phase}] {len(self.file_paths)} samples | classes: {self.classes}')

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        label = self.labels[idx]
        image = cv2.imread(self.file_paths[idx])
        image = image[:, :, ::-1]

        if self.augment:
            if random.uniform(0, 1) > 0.5:
                image = add_g(image)

        if self.transform is not None:
            image = self.transform(image)

        image_flip = transforms.RandomHorizontalFlip(p=1)(image)
        return image, label, idx, image_flip

In [ ]:
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomHorizontalFlip(),
    transforms.RandomErasing(scale=(0.02, 0.25))
])

eval_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

BATCH_SIZE = 32
NUM_WORKERS = 4

print('Loading datasets...')
full_train_dataset = FolderDataset(TRAIN_DATASET_PATH, phase=TRAIN_PHASE,
                                   transform=train_transforms, label_map=TRAIN_MAP)
test_dataset = FolderDataset(TEST_DATASET_PATH, phase=TEST_PHASE,
                             transform=eval_transforms, label_map=TEST_MAP)

if TRAIN_DATASET_NAME in ['RAFDB', 'AffectNET', 'SFEW']:
    train_size = int(0.8 * len(full_train_dataset))
    valid_size = len(full_train_dataset) - train_size

    g = torch.Generator().manual_seed(42)
    train_dataset, _ = random_split(full_train_dataset, [train_size, valid_size], generator=g)

    full_eval_dataset = FolderDataset(TRAIN_DATASET_PATH, phase=TRAIN_PHASE,
                                      transform=eval_transforms, label_map=TRAIN_MAP, augment=False)
    g = torch.Generator().manual_seed(42)
    _, val_dataset = random_split(full_eval_dataset, [train_size, valid_size], generator=g)
else:
    train_dataset = full_train_dataset
    val_dataset = FolderDataset(TRAIN_DATASET_PATH, phase=VALIDATION_PHASE,
                                transform=eval_transforms, label_map=TRAIN_MAP)

train_loader = data.DataLoader(train_dataset, batch_size=BATCH_SIZE,
                               shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = data.DataLoader(val_dataset, batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = data.DataLoader(test_dataset, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')
print(f'Batches - Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

# CLIP features 사전 캐싱 (eval_transforms + train/test 분리)
print('Caching CLIP features (with eval_transforms)...')
clip_cache_train = {}
clip_cache_test = {}

clip_eval_dataset_train = FolderDataset(TRAIN_DATASET_PATH, phase=TRAIN_PHASE,
                                         transform=eval_transforms, label_map=TRAIN_MAP, augment=False)
clip_eval_loader_train = data.DataLoader(clip_eval_dataset_train, batch_size=BATCH_SIZE,
                                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

clip_model.eval()
with torch.no_grad():
    for imgs, labels, idxs, _ in clip_eval_loader_train:
        imgs = imgs.to(device)
        feats = clip_model.encode_image(imgs).float().cpu()
        for i, idx in enumerate(idxs):
            clip_cache_train[idx.item()] = feats[i]
    print(f'  train: done')
    for imgs, labels, idxs, _ in test_loader:
        imgs = imgs.to(device)
        feats = clip_model.encode_image(imgs).float().cpu()
        for i, idx in enumerate(idxs):
            clip_cache_test[idx.item()] = feats[i]
    print(f'  test: done')
print(f'Cached: train={len(clip_cache_train)}, test={len(clip_cache_test)}')

# 메모리 절약: 캐싱 전용 데이터셋 삭제
del clip_eval_dataset_train, clip_eval_loader_train
import gc
gc.collect()
print('Cleaned up caching objects.')

## 7. AU Mask

In [12]:
AU_CODEBOOK = {
    'angry':    [4, 5, 7, 23, 24],
    'disgust':  [9, 15, 16, 25, 26],
    'fear':     [1, 2, 4, 5, 7, 20, 26],
    'happy':    [6, 12],
    'neutral':  [],
    'sad':      [1, 4, 15, 17],
    'surprise': [1, 2, 5, 26, 27],
}

AU_LOCATIONS = {
    1: [(1, 2), (1, 4)], 2: [(1, 1), (1, 5)], 4: [(1, 2), (1, 4)],
    5: [(2, 2), (2, 4)], 6: [(2, 1), (2, 5)], 7: [(2, 2), (2, 4)],
    9: [(3, 3)], 10: [(3, 3)],
    12: [(4, 2), (4, 4)], 15: [(4, 2), (4, 4)], 17: [(5, 3)],
    23: [(4, 3)], 24: [(4, 3)], 25: [(4, 3), (5, 3)],
    26: [(4, 3), (5, 3)], 27: [(4, 3), (5, 3), (6, 3)]
}

SYNONYM_MAP = {'anger': 'angry', 'happiness': 'happy', 'sadness': 'sad', 'surprised': 'surprise'}

def build_au_heatmap(expression_name, grid_size=7, sigma=0.8):
    heatmap = np.zeros((grid_size, grid_size), dtype=np.float32)
    active_aus = AU_CODEBOOK.get(SYNONYM_MAP.get(expression_name, expression_name), [])
    if not active_aus:
        return torch.ones(grid_size, grid_size)
    for au in active_aus:
        for (r, c) in AU_LOCATIONS.get(au, []):
            if 0 <= r < grid_size and 0 <= c < grid_size:
                heatmap[r, c] += 1.0
    heatmap = gaussian_filter(heatmap, sigma=sigma)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()
    return torch.from_numpy(heatmap)

def build_au_heatmap_table(class_list, grid_size=7):
    maps = [build_au_heatmap(c.lower(), grid_size) for c in class_list]
    return torch.stack(maps).to(device)

def au_alignment_loss(spatial_map, targets, au_table):
    act_map = F.relu(spatial_map.mean(dim=1))
    act_flat = act_map.view(act_map.size(0), -1)
    act_norm = act_flat / act_flat.max(dim=1, keepdim=True)[0].clamp(min=1e-6)
    au_guide = au_table[targets].view(targets.size(0), -1)
    cos_sim = F.cosine_similarity(act_norm, au_guide, dim=1)
    return 1.0 - cos_sim.mean()

print('AU Mask functions ready.')

AU Mask functions ready.


## 8. Model Architecture

In [ ]:
class my_MaxPool2d(Module):
    def __init__(self, kernel_size, stride=None, padding=0, dilation=1,
                 return_indices=False, ceil_mode=False):
        super(my_MaxPool2d, self).__init__()
        self.kernel_size    = kernel_size
        self.stride         = stride or kernel_size
        self.padding        = padding
        self.dilation       = dilation
        self.return_indices = return_indices
        self.ceil_mode      = ceil_mode

    def forward(self, input):
        input = input.transpose(3, 1)
        input = F.max_pool2d(input, self.kernel_size, self.stride,
                             self.padding, self.dilation, self.ceil_mode,
                             self.return_indices)
        input = input.transpose(3, 1).contiguous()
        return input


def Mask(nb_batch):
    bar = []
    for i in range(7):
        foo = [1] * 63 + [0] * 10
        if i == 6:
            foo = [1] * 64 + [0] * 10
        random.shuffle(foo)
        bar += foo
    bar = [bar for _ in range(nb_batch)]
    bar = np.array(bar).astype('float32')
    bar = bar.reshape(nb_batch, 512, 1, 1)
    bar = torch.from_numpy(bar).to(device)
    bar = Variable(bar)
    return bar


def supervisor(x, targets, cnum=73):
    branch = x
    branch = branch.reshape(branch.size(0), branch.size(1), 1, 1)
    branch = my_MaxPool2d(kernel_size=(1, cnum), stride=(1, cnum))(branch)
    branch = branch.reshape(branch.size(0), branch.size(1),
                            branch.size(2) * branch.size(3))
    loss_2 = 1.0 - 1.0 * torch.mean(torch.sum(branch, 2)) / cnum

    mask = Mask(x.size(0))
    branch_1 = x.reshape(x.size(0), x.size(1), 1, 1) * mask
    branch_1 = my_MaxPool2d(kernel_size=(1, cnum), stride=(1, cnum))(branch_1)
    branch_1 = branch_1.view(branch_1.size(0), -1)
    loss_1 = nn.CrossEntropyLoss()(branch_1, targets)

    return [loss_1, loss_2]


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=False):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)

        if downsample:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.downsample = None

    def forward(self, x):
        i = x
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        if self.downsample is not None:
            i = self.downsample(i)
        x += i
        return self.relu(x)


class ResNet(nn.Module):
    def __init__(self, block, n_blocks, channels, output_dim):
        super().__init__()
        self.in_channels = channels[0]
        assert len(n_blocks) == len(channels) == 4
        self.conv1   = nn.Conv2d(3, self.in_channels, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1     = nn.BatchNorm2d(self.in_channels)
        self.relu    = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1  = self.get_resnet_layer(block, n_blocks[0], channels[0])
        self.layer2  = self.get_resnet_layer(block, n_blocks[1], channels[1], stride=2)
        self.layer3  = self.get_resnet_layer(block, n_blocks[2], channels[2], stride=2)
        self.layer4  = self.get_resnet_layer(block, n_blocks[3], channels[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(self.in_channels, output_dim)

    def get_resnet_layer(self, block, n_blocks, channels, stride=1):
        layers = []
        downsample = (self.in_channels != block.expansion * channels)
        layers.append(block(self.in_channels, channels, stride, downsample))
        for _ in range(1, n_blocks):
            layers.append(block(block.expansion * channels, channels))
        self.in_channels = block.expansion * channels
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        h = x.view(x.shape[0], -1)
        x = self.fc(h)
        return x, h


class Model(nn.Module):
    def __init__(self, msceleb_path, num_classes=7, drop_rate=0):
        super(Model, self).__init__()
        res18 = ResNet(block=BasicBlock, n_blocks=[2, 2, 2, 2],
                       channels=[64, 128, 256, 512], output_dim=1000)
        msceleb_model = torch.load(msceleb_path, map_location='cpu', weights_only=False)
        state_dict = msceleb_model['state_dict']
        res18.load_state_dict(state_dict, strict=False)
        print('MS-Celeb weights loaded.')

        self.drop_rate = drop_rate
        self.features  = nn.Sequential(*list(res18.children())[:-2])
        self.features2 = nn.Sequential(*list(res18.children())[-2:-1])
        fc_in_dim = list(res18.children())[-1].in_features
        self.fc = nn.Linear(fc_in_dim, num_classes)

        self.parm = {}
        for name, parameters in self.fc.named_parameters():
            print(f'  {name}: {parameters.size()}')
            self.parm[name] = parameters

    def forward(self, x, clip_features, targets, phase='train'):
        image_features = clip_features.to(x.device)

        spatial_map = self.features(x)
        x = self.features2(spatial_map)
        x = x.view(x.size(0), -1)

        masked_features = image_features * torch.sigmoid(x)
        out = self.fc(masked_features)

        if phase == 'train':
            MC_loss = supervisor(masked_features, targets, cnum=73)
            return out, MC_loss, spatial_map
        else:
            return out, out

print('Model architecture defined.')

## 9. Initialize Model, Optimizer, Scheduler

In [14]:
setup_seed(3407)

model = Model(msceleb_path=MSCELEB_PATH, num_classes=NUM_CLASSES)
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0002, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

EPOCHS = 20

print(f'\nOptimizer : Adam  lr=0.0002, wd=1e-4')
print(f'Scheduler : ExponentialLR gamma=0.9')
print(f'Epochs    : {EPOCHS}')

MS-Celeb weights loaded.
  weight: torch.Size([7, 512])
  bias: torch.Size([7])

Optimizer : Adam  lr=0.0002, wd=1e-4
Scheduler : ExponentialLR gamma=0.9
Epochs    : 20


## 10. Train & Test Functions

In [ ]:
label_counts = Counter(full_train_dataset.labels)
counts = torch.tensor([label_counts[i] for i in range(NUM_CLASSES)], dtype=torch.float)
class_weights = counts.sum() / (NUM_CLASSES * counts)
class_weights = torch.clamp(class_weights, max=3.0).to(device)

def train_one_epoch(model, train_loader, optimizer, scheduler, device, AU_TABLE):
    model.train()
    running_loss = 0.0
    correct_sum = 0

    for imgs, labels, idxs, _ in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        clip_feats = torch.stack([clip_cache_train[i.item()] for i in idxs])
        output, MC_loss, spatial_map = model(imgs, clip_feats, labels, phase='train')

        loss_cls = nn.CrossEntropyLoss(weight=class_weights)(output, labels)
        loss_au  = au_alignment_loss(spatial_map, labels, AU_TABLE)
        loss = loss_cls + 5 * MC_loss[1] + 1.5 * MC_loss[0] + 0.5 * loss_au

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, predicts = torch.max(output, 1)
        correct_sum += torch.eq(predicts, labels).sum()
        running_loss += loss.item()

    scheduler.step()
    return (correct_sum.float() / len(train_loader.dataset)).item(), running_loss / len(train_loader)


def test_model(model, loader, device, cache):
    model.eval()
    with torch.no_grad():
        running_loss = 0.0
        iter_cnt = 0
        correct_sum = 0
        data_num = 0

        for imgs, labels, idxs, imgs_flip in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            clip_feats = torch.stack([cache[i.item()] for i in idxs])
            outputs, _ = model(imgs, clip_feats, labels, phase='test')
            loss = nn.CrossEntropyLoss()(outputs, labels)

            iter_cnt += 1
            _, predicts = torch.max(outputs, 1)
            correct_sum += torch.eq(predicts, labels).sum()
            running_loss += loss
            data_num += outputs.size(0)

        running_loss = running_loss / iter_cnt
        test_acc = correct_sum.float() / float(data_num)

    return test_acc.item(), running_loss.item()

print('Train and test functions ready.')
print(f'Batch size: {BATCH_SIZE} | Train batches: {len(train_loader)}')

## 11. Training Loop

In [ ]:
# AU Table (label index 순서로 생성)
AU_TABLE = build_au_heatmap_table(LABEL_NAMES)
print(f'AU Table ready for: {LABEL_NAMES}')

history = {'train_acc': [], 'train_loss': [], 'val_acc': [], 'val_loss': []}
best_acc = 0.0
patience = 5
patience_counter = 0

print('=' * 60)
print(f'Training CAFE on {TRAIN_DATASET_NAME}')
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')
print('=' * 60)

for epoch in range(1, EPOCHS + 1):

    train_acc, train_loss = train_one_epoch(
        model, train_loader, optimizer, scheduler, device, AU_TABLE
    )
    val_acc, val_loss = test_model(model, val_loader, device, clip_cache_train)

    history['train_acc'].append(train_acc)
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)
    history['val_loss'].append(val_loss)

    print(f'Epoch [{epoch:>3}/{EPOCHS}] '
          f'Train Acc: {train_acc*100:5.2f}% Loss: {train_loss:.4f} | '
          f'Val Acc: {val_acc*100:5.2f}% Loss: {val_loss:.4f}')

    if val_acc > best_acc:
        best_acc = val_acc
        patience_counter = 0
        torch.save({'model_state_dict': model.state_dict()},
                    os.path.join(SAVE_DIR, f'ours_best_{TRAIN_DATASET_NAME}.pth'))
        print(f'  -> Best model saved (acc={best_acc*100:.2f}%)')
    else:
        patience_counter += 1
        print(f'  -> No improvement ({patience_counter}/{patience})')

    if patience_counter >= patience:
        print(f'Early stopping at epoch {epoch} (no improvement for {patience} epochs)')
        break

    torch.save({'model_state_dict': model.state_dict()},
                os.path.join(SAVE_DIR, f'ours_final_{TRAIN_DATASET_NAME}.pth'))

print('=' * 60)
print(f'Training complete. Best val acc: {best_acc*100:.2f}%')

## 12. Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_x = list(range(1, len(history['train_acc']) + 1))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_x, [a * 100 for a in history['train_acc']], 'b-o', markersize=3, label='Train')
axes[0].plot(epochs_x, [a * 100 for a in history['val_acc']],  'r-o', markersize=3, label='Val')
axes[0].set_title(f'Accuracy - {TRAIN_DATASET_NAME}')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy (%)')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_x, history['train_loss'], 'b-o', markersize=3, label='Train')
axes[1].plot(epochs_x, history['val_loss'],  'r-o', markersize=3, label='Val')
axes[1].set_title(f'Loss - {TRAIN_DATASET_NAME}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()
print(f'Best val accuracy: {best_acc * 100:.2f}%')

## 13. Load Best Model & Final Test

In [ ]:
print(f'Evaluating on test set: {TEST_DATASET_NAME}')

model_path = os.path.join(SAVE_DIR, f'ours_best_{TRAIN_DATASET_NAME}.pth')
checkpoint = torch.load(model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

test_acc, test_loss = test_model(model, test_loader, device, clip_cache_test)
print(f'Final Test Acc: {test_acc*100:.2f}% Loss: {test_loss:.4f}')

## 14. Per-Class Accuracy & Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels, idxs, image_flip in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        clip_feats = torch.stack([clip_cache_test[i.item()] for i in idxs])
        outputs, _ = model(images, clip_feats, labels, phase='test')
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

class_names = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

print(classification_report(all_labels, all_preds, target_names=class_names))

fig, ax = plt.subplots(figsize=(8, 8))
matrix = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(all_labels, all_preds),
    display_labels=class_names
)
matrix.plot(cmap='Blues', xticks_rotation=90, ax=ax)
ax.set_title(f'Confusion Matrix - Train:{TRAIN_DATASET_NAME} Test:{TEST_DATASET_NAME}')
plt.tight_layout()
plt.show()

## 15. Sanity Check

In [ ]:
print('Running sanity check...')
model.eval()

dummy_clip   = torch.randn(4, 512).to(device)
dummy_imgs   = torch.randn(4, 3, 224, 224).to(device)
dummy_labels = torch.randint(0, 7, (4,)).to(device)

with torch.no_grad():
    out, _ = model(dummy_imgs, dummy_clip, dummy_labels, phase='test')
print(f'Output shape (test) : {out.shape}  (expected: [4, 7])')

model.train()
out, MC_loss, spatial_map = model(dummy_imgs, dummy_clip, dummy_labels, phase='train')
l_cls = nn.CrossEntropyLoss()(out, dummy_labels)
total = l_cls + 5 * MC_loss[1] + 1.5 * MC_loss[0]

print(f'Output shape (train): {out.shape}  (expected: [4, 7])')
print(f'l_cls  : {l_cls.item():.4f}')
print(f'l_sep  : {MC_loss[0].item():.4f}')
print(f'l_div  : {MC_loss[1].item():.4f}')
print(f'total  : {total.item():.4f}')
print('\nSanity check passed!')